# The $(p_+,p_-)$ triplet algebra $\mathcal{M}_{p_+,p_-}$

For coprime integers $p_+, p_- \ge 2$, the *$(p_+,p_-)$ triplet algebra* $\mathcal{M}_{p_+,p_-}$ is a logarithmic extension of the Virasoro minimal model of central charge $c_{p_+,p_-} = 1 - 6(p_+ - p_-)^2/(p_+p_-)$ [FGST2006]. This notebook follows Tsuchiya and Wood [TW2013], who call it the extended $W$-algebra of type $\mathfrak{sl}_2$ at positive rational level.

$\mathcal{M}_{p_+,p_-}$ is the subalgebra of the lattice vertex algebra $V_{p_+,p_-}$ of the lattice $Y=\mathbb{Z}\sqrt{2p_+p_-}$ given by the intersection of the kernels of two screening operators. It is strongly generated by the Virasoro field $T$, built from a Heisenberg field with a background charge, and three fields $W^-, W^0, W^+$ of conformal weight $\Delta_1=(2p_+-1)(2p_--1)$ [TW2013, Thm 5.4, Rem 5.5]. The $W$ fields are given by integrated products of screening operators, which [TW2013, Prop 3.24] evaluates as Jack symmetric functions [Mac1995] in the Heisenberg modes.

This notebook builds $T, W^-, W^0, W^+$ in the library's lattice vertex algebra, checks them, constructs the vertex subalgebra they generate (which verifies that they strongly generate it), and computes the OPEs of the $W$ fields in terms of the strong generators. The default $(p_+, p_-) = (2, 3)$ gives the triplet algebra of central charge $c = 0$ [GRW2009]. **The parameters `pp`, `pm` of section 1 can be changed to any coprime pair $\ge 2$.**

**Prerequisites:** `05_lattice_vertex_algebras.ipynb`, `06_subalgebras_ideals_quotients.ipynb`.

**Elsewhere:** the triplet algebra $\mathcal W(2)$ at $c = -2$, with its modules, fusion products and extensions, is in `15_triplet_modules.ipynb`; Virasoro fusion at $c = 0$, the central charge of $\mathcal M_{2,3}$, is in `16_virasoro_c0_fusion.ipynb`.

**Contents**

1. Parameters
2. The lattice vertex algebra and the dictionary to [TW2013]
3. Jack polynomials in power sums
4. Screening operators
5. The soliton vector $W_{0,0}$ and the $W$ fields
6. The vertex subalgebra generated by $T, W^-, W^0, W^+$
7. OPEs
8. Checks: no vacuum terms, nothing below weight $\Delta_0$, poles in the ideal of $W_{0,0}$
9. Normalisation
10. Limitations

In [1]:
import vertex_algebras as va
from vertex_algebras import *
import os, sys, time
from vertex_algebras.lattice_vertex_algebra import LatticeVertexAlgebra
from vertex_algebras.vertex_algebra import VertexAlgebraParallelism

## 1. Parameters

The cell below sets the parameters, and computes the weight $\Delta_0 = (p_+-1)(p_--1)$ of the soliton vector $W_{0,0}$ (section 5), the weight $\Delta_1$ of the $W$ fields, and the central charge.

* `pp`, `pm` are $p_+$ and $p_-$.
* `NCPUS` is the number of threads of the library's C++ engine, a setting of the session (`VertexAlgebraParallelism().set(NCPUS)`). The threads are used for the heavy steps: the large $W$–$W$ OPEs, and the normally ordered monomials that span the weight spaces of high weight (one engine call per weight space). Small products stay serial.
* `DATA_DIR` is the directory of the saved results, and `RECOMPUTE` decides whether to ignore them (see below).

**Running time.** For $(p_+,p_-)=(2,3)$, on a workstation with 28 cores and 64 GB of RAM, the strong generation check (spans up to weight $2\Delta_1-1=29$) takes about 2 minutes and about 5 GB of RAM with the settings below, saving its result takes 15 seconds, and each $W$–$W$ OPE takes a few seconds. Other pairs have larger $\Delta_1$ and take considerably longer.

**Saved results.** The vertex subalgebra (its strong generators and the spans of normally ordered monomials, `save(M, file)`) and the $W$–$W$ OPEs are saved in `DATA_DIR` and loaded from there on later runs, so the expensive steps run only once. The files do not depend on `NCPUS` or on the compiled C++ engines, so `DATA_DIR` can be copied from a machine that computed them, to run this notebook on a smaller machine in minutes. Loading checks that the files belong to the same algebra and generators, but it does not repeat the strong generation check. Setting `RECOMPUTE = True` ignores the files, and computes and saves everything again. On the workstation above, a run with the saved files takes about half a minute (loading the 100 MB state takes about 6 s and 3 GB of RAM); the outputs below are from a run that loaded the saved subalgebra. The files are Sage pickles, so only files from a trusted source should be loaded.

In [2]:
pp, pm = 2, 3          # p_+, p_-: coprime integers >= 2
NCPUS = min(24, os.cpu_count())   # engine threads
VertexAlgebraParallelism().set(NCPUS)
DATA_DIR = 'triplet_data'   # saved subalgebra state and OPEs
RECOMPUTE = False      # True: ignore saved files, compute and save again
assert pp >= 2 and pm >= 2 and gcd(pp, pm) == 1, 'p_+, p_- must be coprime and >= 2'
Delta0 = (pp - 1) * (pm - 1)          # weight of the soliton vector W_{0,0}
Delta1 = (2*pp - 1) * (2*pm - 1)      # weight of the W fields
c_expected = 1 - 6 * (pp - pm)^2 / (pp * pm)
os.makedirs(DATA_DIR, exist_ok=True)
STATE_FILE = os.path.join(DATA_DIR, 'triplet_%d_%d_subalgebra.sobj' % (pp, pm))
OPE_FILE = os.path.join(DATA_DIR, 'triplet_%d_%d_opes.sobj' % (pp, pm))
print('Delta_0 =', Delta0, ' Delta_1 =', Delta1, ' c =', c_expected)

Delta_0 = 2  Delta_1 = 15  c = 0


## 2. The lattice vertex algebra and the dictionary to [TW2013]

Tsuchiya and Wood [TW2013, §2] use a Heisenberg field $b(z)$ with $b(z)b(w)\sim (z-w)^{-2}$, Heisenberg weights $\beta$ (so that $b_0|\beta\rangle=\beta|\beta\rangle$), and the constants

$$\alpha_+=\sqrt{2p_-/p_+},\quad \alpha_-=-\sqrt{2p_+/p_-},\quad
\alpha_0=\alpha_++\alpha_-,\quad \alpha=p_+\alpha_+=-p_-\alpha_-=\sqrt{2p_+p_-},\quad
\kappa_\pm=\alpha_\pm^2/2 .$$

The library's lattice vertex algebra of the Gram matrix $(2p_+p_-)$ has one current $J$, with $J(z)J(w)\sim 2p_+p_-(z-w)^{-2}$, and the states $e^{nx}$, $n \in \mathbb Z$. The dictionary is:

* $b = J/\alpha$ and $|n\alpha\rangle = e^{nx}$, so Heisenberg weights are measured in units of $\alpha$, in which $\alpha_+ = 1/p_+$ and $\alpha_- = -1/p_-$;
* $\beta_{r,s} = \frac{1-r}{2}\alpha_+ + \frac{1-s}{2}\alpha_-$, that is, $\beta_{r,s}/\alpha = \frac{1-r}{2p_+}-\frac{1-s}{2p_-}$;
* $T=\tfrac12({:}bb{:}+\alpha_0\partial b)$ [TW2013, Def 2.24], that is, $T = \frac{1}{4p_+p_-}{:}JJ{:} + \frac{p_--p_+}{2p_+p_-}\partial J$, with rational coefficients.

The cell below builds the lattice vertex algebra with this conformal vector and with the lattice grading, in which $e^{nx}$ has grade $n$ (`05_lattice_vertex_algebras.ipynb`, section 5). It checks the central charge and prints the conformal weights of the generators: $e^{nx}$ has weight $p_+p_-n^2 - (p_- - p_+)n$, which is $5$ and $7$ for $n = \pm 1$ at $(p_+, p_-) = (2, 3)$.

In [3]:
B = 2 * pp * pm
L = LatticeVertexAlgebra(QQ, matrix([[B]]))
J = L.J(0)
T = J.no(J) / (2*B) + QQ(pm - pp) / B * J.derivative()
L = L.with_conformal_vector(T).with_horizontal_grading('lattice')   # grading: the lattice charge n of e^{nx}
J, T = L.J(0), L.conformal_vector()
assert L.central_charge() == c_expected
L.central_charge(), L.degrees()

(0, {'J0': 1, 'e^(1)': 5, 'e^(-1)': 7})

## 3. Jack polynomials in power sums

[TW2013, §3.3] uses the Jack polynomials $P_\lambda(x;\kappa)$ in Macdonald's normalisation [Mac1995, Ch. VI, §10], which are orthogonal for the inner product $\langle p_\lambda, p_\mu\rangle_\kappa=\delta_{\lambda\mu}z_\lambda\kappa^{\ell(\lambda)}$ on power sums. Sage's built-in `jack(t=...)` bases compute the transition matrix for a *symbolic* parameter on all partitions of the degree, which is far too slow here. The function `jack_P` in the cell below computes $P_\lambda$ directly instead: $P_\lambda = m_\lambda + \sum_{\mu<\lambda} u_\mu m_\mu$, with the sum over the partitions $\mu$ below $\lambda$ in the dominance order, is the element orthogonal to all $m_\mu$ with $\mu<\lambda$, which takes one linear solve over $\mathbb{Q}$. The normalisation of the $W$ fields is arbitrary anyway, so $P_\lambda$ is used rather than its dual $Q_\lambda$. The cell checks `jack_P` against Sage's implementation on three small partitions, with $\kappa = p_+/p_-$.

In [4]:
Sym = SymmetricFunctions(QQ); p_basis = Sym.p(); m_basis = Sym.m()

def jack_P(lam, kappa):
    '''Macdonald's Jack polynomial P_lam(x; kappa) in power sums, as a dict
    {partition: coefficient}.'''
    lam = Partition(lam); n = lam.size()
    below = [mu for mu in Partitions(n) if lam.dominates(mu) and mu != lam]
    pexp = lambda mu: p_basis(m_basis[mu]).monomial_coefficients()
    ip = lambda f, g: sum(c * g.get(r, 0) * Partition(r).centralizer_size() * kappa^len(r)
                          for r, c in f.items())
    E = [pexp(mu) for mu in below]; P = dict(pexp(lam))
    if E:
        G = matrix(QQ, [[ip(a, b) for b in E] for a in E])
        u = G.solve_right(-vector(QQ, [ip(P, b) for b in E]))
        for c, a in zip(u, E):
            for r, v in a.items():
                P[r] = P.get(r, 0) + c * v
    return {r: c for r, c in P.items() if c}

# check against Sage's implementation on small partitions
for lam in [[2, 1], [2, 2], [3, 1, 1]]:
    ref = p_basis(Sym.jack(t=QQ(pp)/pm).P()[lam]).monomial_coefficients()
    assert {tuple(k): v for k, v in ref.items()} == {tuple(k): v for k, v in jack_P(lam, QQ(pp)/pm).items()}
print('jack_P agrees with Sage')

jack_P agrees with Sage


## 4. Screening operators

For $r\ge 1$ and $s\le 0$, [TW2013, Prop 3.24] gives

$$Q_+^{[r]}|\beta_{r,s}\rangle = \rho_{2/\alpha_+}\big(Q_{\lambda_{r,-s}}(x;\kappa_-)\big)|\beta_{-r,s}\rangle,$$

where $\lambda_{N,M}=(M,\dots,M)$ is the rectangular partition with $N$ parts, and $\rho_\gamma$ is the map $p_n \mapsto \gamma\, b_{-n}$ from power sums to Heisenberg modes. Since $\frac{2}{\alpha_+}b_{-n} = \frac{2}{\alpha_+\alpha}J_{(-n)} = \frac{1}{p_-}J_{(-n)}$, the power sum $p_n$ becomes $\frac{1}{p_-}J_{(-n)}$, a mode acting on $e^{qx}$. The function `screened(r, s)` in the cell below computes $Q_+^{[r]}|\beta_{r,s}\rangle$ in this way, with $\kappa = \kappa_- = p_+/p_-$, using $P_\lambda$ instead of $Q_\lambda=b_\lambda P_\lambda$; the two differ by the nonzero constant $b_\lambda$.

In [5]:
def lattice_charge(r, s):
    '''beta_{r,s} in units of alpha (must be an integer for a lattice state).'''
    q = QQ(1 - r) / (2*pp) - QQ(1 - s) / (2*pm)
    assert q in ZZ, (r, s, q)
    return ZZ(q)

def screened(r, s, kappa=QQ(pp)/pm):
    '''Q_+^{[r]} |beta_{r,s}> (r >= 1, s <= 0) up to a nonzero constant.'''
    lam = [-s] * r
    q = lattice_charge(-r, s)
    cache = {(): L.e((q,))}
    def apply(mu):              # prod_i (1/p_-) J_(-mu_i) applied to e^{qx}
        if mu not in cache:
            cache[mu] = J.nproduct(apply(mu[1:]), -mu[0]) / pm
        return cache[mu]
    return sum((c * apply(tuple(mu)) for mu, c in jack_P(lam, kappa).items()), L.zero())

## 5. The soliton vector $W_{0,0}$ and the $W$ fields

The fields $W_{n,m}=E^{n+m}Q_+^{[p_+-1]}|\alpha_n^+\rangle$, with $\alpha_n^+=\beta_{p_+-1,1;-2n-1}$, are defined in [TW2013, Def 5.2], where $E$ is the Frobenius map. By [TW2013, Thm 4.17(2)], $E$ maps $Q_+^{[(k+1)p_+-1]}|\beta\rangle$ to a nonzero multiple of $Q_+^{[(k+2)p_+-1]}|\beta\rangle$. Hence, up to nonzero constants:

* $W_{0,0} = Q_+^{[p_+-1]}|\alpha_0^+\rangle$, the *soliton vector*, of weight $\Delta_0=(p_+-1)(p_--1)$ and charge 0;
* $W^- = W_{1,-1} = Q_+^{[p_+-1]}|\alpha_1^+\rangle$, $W^0 = W_{1,0}\propto Q_+^{[2p_+-1]}|\alpha_1^+\rangle$ and $W^+ = W_{1,1}\propto Q_+^{[3p_+-1]}|\alpha_1^+\rangle$, of weight $\Delta_1$ and charges $-1, 0, 1$.

Writing $\alpha_n^+=(1-(n+1)p_+)\alpha_+$ as $\beta_{r,s}$ gives $s = 1 + p_-(r+1-2(n+1)p_+)/p_+$; this is the function `s_of(r, n)` below. For example, $\alpha_0^+=\beta_{p_+-1,1-p_-}$. The cell below builds the four vectors, checks their weights and charges and that they are Virasoro primary ($L_1$ and $L_2$ kill them), and prints the number of terms of each.

In [6]:
def s_of(r, n):
    '''s with beta_{r,s} = alpha_n^+ = (1 - (n+1) p_+) alpha_+.'''
    s = 1 + QQ(pm) * (r + 1 - 2*(n + 1)*pp) / pp
    assert s in ZZ
    return ZZ(s)

t0 = time.time()
W00 = screened(pp - 1, s_of(pp - 1, 0))
Wm  = screened(pp - 1, s_of(pp - 1, 1))
W0  = screened(2*pp - 1, s_of(2*pp - 1, 1))
Wp  = screened(3*pp - 1, s_of(3*pp - 1, 1))
print('built in %.1fs' % (time.time() - t0))
for name, W, h, q in [('W00', W00, Delta0, 0), ('W-', Wm, Delta1, -1), ('W0', W0, Delta1, 0), ('W+', Wp, Delta1, 1)]:
    primary = all(T.nproduct(W, k).is_zero() for k in (2, 3))     # L_1 W = L_2 W = 0
    print(name, ' weight', W.degree(), ' charge', W.horizontal_grade(), ' primary', primary,
          ' terms', len(W.support()))
    assert W.degree() == h and W.horizontal_grade() == q and primary

built in 1.8s
W00  weight 2  charge 0  primary True  terms 2
W-  weight 15  charge -1  primary True  terms 22
W0  weight 15  charge 0  primary True  terms 172
W+  weight 15  charge 1  primary True  terms 42


$W_{0,0}$ is a Virasoro descendant of the vacuum [TW2013, proof of Thm 5.4]. The cell below checks that $W_{0,0}$ lies in the subalgebra generated by $T$, and expresses it in $T$; for $(p_+, p_-) = (2, 3)$, $\Delta_0 = 2$ and $W_{0,0}$ is a multiple of $T$.

In [7]:
Vir = L.subalgebra([T], names=['T'])
Vir.print_options(generators='strong')
assert W00 in Vir
Vir.express(W00)

8/5*T

## 6. The vertex subalgebra generated by $T, W^-, W^0, W^+$

The constructor closes $\{T,W^-,W^0,W^+\}$ under all $n$-th products, $n\ge0$ (`06_subalgebras_ideals_quotients.ipynb`, section 1). With `max_closure_steps=1`, it checks that one round of the closure adds no new strong generator, that is, that these four fields strongly generate the subalgebra [TW2013, Rem 5.5]. The products $W^\pm_{(n)}W^\mp$ reach weight $2\Delta_1-1$, so this is the expensive step. With `horizontal_grading='ambient'`, the closure works separately in each sector of the lattice charge (the grading set on `L` in section 2), which keeps the spans small.

If `STATE_FILE` exists and `RECOMPUTE` is `False`, the cell below loads the subalgebra from the file (`load`) instead. The file records the strong generators and spans of a closure computed earlier with the same parameters, and the constructor then returns the loaded subalgebra without closing again. Otherwise, including when the file is for other parameters, the closure runs and its result is saved (`save(M, file)`). The cell ends by expressing $W_{0,0}$ in the strong generators.

In [8]:
t0 = time.time()
gens, names = [T, Wm, W0, Wp], ['T', 'Wm', 'W0', 'Wp']
M = None
if os.path.exists(STATE_FILE) and not RECOMPUTE:
    M = load(STATE_FILE)
    if not (hasattr(M, 'ambient') and M.ambient() is L
            and list(M.variable_names()[:4]) == names):
        M = None          # a file of other parameters (or of an older version)
if M is not None:
    assert L.subalgebra(gens, names=names, max_closure_steps=1,
                        horizontal_grading='ambient') is M      # no closure again
    print('loaded from %s in %.1fs:' % (STATE_FILE, time.time() - t0), M.variable_names())
else:
    M = L.subalgebra(gens, names=names, max_closure_steps=1, horizontal_grading='ambient')
    print('strong generation verified in %.1fs:' % (time.time() - t0), M.variable_names())
    save(M, STATE_FILE)
    print('saved to %s (%.0f MB)' % (STATE_FILE, os.path.getsize(STATE_FILE) / 2^20))
M.print_options(generators='strong')
T_, Wm_, W0_, Wp_ = M.gens()
W00_ = M(W00)
print('W00 =', W00_)

loaded from triplet_data/triplet_2_3_subalgebra.sobj in 7.1s: ('T', 'Wm', 'W0', 'Wp')
W00 = 8/5*T


## 7. OPEs

`a.ope(b)[n]` is $a_{(n)}b$, the coefficient of $(z-w)^{-n-1}$, of conformal weight $h_a+h_b-n-1$; it is printed in terms of the strong generators. By [TW2013, Thm 5.4], the $W$ fields lie in the ideal $X_{1,1}$ generated by $W_{0,0}$. Hence **no pole of a $W$–$W$ OPE has a vacuum term, and the poles of weight $<\Delta_0$ vanish**; section 8 checks this.

The function `show_ope` in the cell below prints the poles of an OPE, and summarises long poles by their number of terms (set `SHOW_ALL = True` to print everything). Each OPE is computed once and saved in `OPE_FILE`, as dicts over the basis of the lattice vertex algebra, and later runs load it from there. The cell prints the OPE of $T$ with $W^+$, which shows that $W^+$ is primary of weight $\Delta_1$.

In [9]:
SHOW_ALL = False
MAX_TERMS = 12

# saved OPEs: {(na, nb): [pole as a dict over the basis keys of L]}, plus the
# generators they were computed for (a changed normalisation discards them)
GENS = [g.monomial_coefficients() for g in gens]
OPES = load(OPE_FILE) if os.path.exists(OPE_FILE) and not RECOMPUTE else {}
if OPES.get('generators') != GENS:
    OPES = {'generators': GENS}

def ope_of(a, b, na, nb):
    '''a.ope(b) for strong generators a, b of M, from OPE_FILE if saved there.'''
    key = (na, nb)
    if key not in OPES:
        OPES[key] = [x.monomial_coefficients() for x in a.ope(b)]
        save(OPES, OPE_FILE)
    return [M.sum_of_terms(d.items()) for d in OPES[key]]

def show_ope(a, b, na, nb):
    poles = ope_of(a, b, na, nb)
    ha, hb = M.degree(a), M.degree(b)
    print('%s(z) %s(w): %d poles' % (na, nb, len(poles)))
    for n in range(len(poles) - 1, -1, -1):
        x = poles[n]
        if x.is_zero():
            continue
        e = M.express(x)
        terms = len(e.monomial_coefficients())
        s = repr(e) if (SHOW_ALL or terms <= MAX_TERMS) else '<%d terms>' % terms
        print('  (z-w)^%-3d weight %-3s %s' % (-n - 1, ha + hb - n - 1, s))
    return poles

_ = show_ope(T_, Wp_, 'T', 'W+')

T(z) W+(w): 2 poles
  (z-w)^-2  weight 15  15*Wp
  (z-w)^-1  weight 16  dWp


The OPE of $W^+$ with $W^-$ has $2\Delta_1 - \Delta_0 = 28$ poles, of weights $2, \dots, 29$. The pole of highest order has weight $\Delta_0 = 2$ and is a multiple of $T$, which is a multiple of $W_{0,0}$ here; the poles of weight up to 10 are printed in full.

In [10]:
t0 = time.time()
ope_pm = show_ope(Wp_, Wm_, 'W+', 'W-')
print('%.1fs' % (time.time() - t0))

W+(z) W-(w): 28 poles
  (z-w)^-28 weight 2   262144/12155*T
  (z-w)^-27 weight 3   131072/12155*dT
  (z-w)^-26 weight 4   9961472/133705*:T T: - 196608/10285*d^2T
  (z-w)^-25 weight 5   9961472/133705*:dT T: - 131072/30855*d^3T
  (z-w)^-24 weight 6   -28573696/524535*:T T T: - 30736384/174845*:dT dT: + 103153664/454597*:d^2T T: + 2818048/524535*d^4T
  (z-w)^-23 weight 7   -14286848/174845*:dT T T: - 14155776/174845*:d^2T dT: + 638451712/6818955*:d^3T T: + 704512/874225*d^5T
  (z-w)^-22 weight 8   11927552/524535*:T T T T: + 19267584/174845*:dT dT T: - 32571392/174845*:d^2T T T: + 17891328/174845*:d^2T d^2T: - 19955712/174845*:d^3T dT: + 230752256/6818955*:d^4T T: - 802816/2622675*d^6T
  (z-w)^-21 weight 9   23855104/524535*:dT T T T: + 655360/9537*:dT dT dT: - 6160384/174845*:d^2T dT T: - 2621440/34969*:d^3T T T: + 23855104/524535*:d^3T d^2T: - 24838144/524535*:d^4T dT: + 392200192/34094775*:d^5T T: - 1015808/15020775*d^7T
  (z-w)^-20 weight 10  -1933312/238425*:T T T T T: - 7077888/17

  (z-w)^-12 weight 18  <91 terms>
  (z-w)^-11 weight 19  <110 terms>


  (z-w)^-10 weight 20  <144 terms>
  (z-w)^-9  weight 21  <176 terms>
  (z-w)^-8  weight 22  <224 terms>


  (z-w)^-7  weight 23  <274 terms>


  (z-w)^-6  weight 24  <348 terms>


  (z-w)^-5  weight 25  <422 terms>


  (z-w)^-4  weight 26  <528 terms>


  (z-w)^-3  weight 27  <643 terms>


  (z-w)^-2  weight 28  <796 terms>


  (z-w)^-1  weight 29  <964 terms>
7.7s


The OPE of $W^0$ with itself has the same pole structure:

In [11]:
ope_00 = show_ope(W0_, W0_, 'W0', 'W0')

W0(z) W0(w): 28 poles
  (z-w)^-28 weight 2   -1048576/45815*T
  (z-w)^-27 weight 3   -524288/45815*dT
  (z-w)^-26 weight 4   -39845888/503965*:T T: + 10223616/503965*d^2T
  (z-w)^-25 weight 5   -39845888/503965*:dT T: + 6815744/1511895*d^3T
  (z-w)^-24 weight 6   1485832192/25702215*:T T T: + 228327424/1223915*:dT dT: - 412614656/1713481*:d^2T T: - 146538496/25702215*d^4T
  (z-w)^-23 weight 7   742916096/8567405*:dT T T: + 736100352/8567405*:d^2T dT: - 2553806848/25702215*:d^3T T: - 36634624/42837025*d^5T
  (z-w)^-22 weight 8   -88604672/3671745*:T T T T: - 20447232/174845*:dT dT T: + 241958912/1223915*:d^2T T T: - 132907008/1223915*:d^2T d^2T: + 148242432/1223915*:d^3T dT: - 131858432/3671745*:d^4T T: + 851968/2622675*d^6T
  (z-w)^-21 weight 9   -177209344/3671745*:dT T T T: - 34078720/467313*:dT dT dT: + 320339968/8567405*:d^2T dT T: + 136314880/1713481*:d^3T T T: - 177209344/3671745*:d^3T d^2T: + 1291583488/25702215*:d^4T dT: - 1568800768/128511075*:d^5T T: + 52822016/736017975*d^7T

  (z-w)^-7  weight 23  <253 terms>
  (z-w)^-6  weight 24  <320 terms>


  (z-w)^-5  weight 25  <383 terms>


  (z-w)^-4  weight 26  <478 terms>


  (z-w)^-3  weight 27  <574 terms>


  (z-w)^-2  weight 28  <708 terms>


  (z-w)^-1  weight 29  <847 terms>


The remaining OPEs: $W^0$ with $W^\pm$ has 15 poles, the pole of highest order being a multiple of $W^\pm$, and the OPEs of $W^+$ with $W^+$ and of $W^-$ with $W^-$ are regular.

In [12]:
ope_0p = show_ope(W0_, Wp_, 'W0', 'W+')
ope_0m = show_ope(W0_, Wm_, 'W0', 'W-')
ope_pp = show_ope(Wp_, Wp_, 'W+', 'W+')
ope_mm = show_ope(Wm_, Wm_, 'W-', 'W-')

W0(z) W+(w): 15 poles
  (z-w)^-15 weight 15  447488/11*Wp
  (z-w)^-14 weight 16  223744/11*dWp
  (z-w)^-13 weight 17  739328/55*:T Wp: + 252928/55*d^2Wp
  (z-w)^-12 weight 18  369664/55*:dT Wp: + 369664/55*:dWp T: - 2118272/165*d^3Wp
  (z-w)^-11 weight 19  109138432/45815*:T T Wp: + 182171392/45815*:dT dWp: - 24981504/45815*:d^2T Wp: + 67084288/45815*:d^2Wp T: - 177729344/137445*d^4Wp
  (z-w)^-10 weight 20  109138432/45815*:dT T Wp: + 94848/2695*:d^2T dWp: - 87170176/137445*:d^3T Wp: + 54569216/45815*:dWp T T: + 4331392/4165*:d^2Wp dT: - 57380608/12495*:d^3Wp T: + 30888832/45815*d^5Wp
  (z-w)^-9  weight 21  78567424/237405*:T T T Wp: + 107374592/870485*:dT dT Wp: + 34464768/20825*:dT dWp T: - 519793664/870485*:d^2T T Wp: + 742929408/4352425*:d^2T d^2Wp: - 308781824/687225*:d^3T dWp: + 1874944/137445*:d^4T Wp: + 998473216/4352425*:d^2Wp T T: - 221341952/69825*:d^3Wp dT: - 23129088/56525*:d^4Wp T: + 9058221952/13057275*d^6Wp
  (z-w)^-8  weight 22  <14 terms>
  (z-w)^-7  weight 23  <21 te

  (z-w)^-2  weight 28  <88 terms>


  (z-w)^-1  weight 29  <117 terms>
W0(z) W-(w): 15 poles
  (z-w)^-15 weight 15  -447488/11*Wm
  (z-w)^-14 weight 16  -223744/11*dWm
  (z-w)^-13 weight 17  -739328/55*:T Wm: - 252928/55*d^2Wm
  (z-w)^-12 weight 18  -369664/55*:dT Wm: - 369664/55*:dWm T: + 2118272/165*d^3Wm
  (z-w)^-11 weight 19  -109138432/45815*:T T Wm: - 182171392/45815*:dT dWm: + 24981504/45815*:d^2T Wm: - 67084288/45815*:d^2Wm T: + 177729344/137445*d^4Wm
  (z-w)^-10 weight 20  -109138432/45815*:dT T Wm: - 94848/2695*:d^2T dWm: + 87170176/137445*:d^3T Wm: - 54569216/45815*:dWm T T: - 4331392/4165*:d^2Wm dT: + 57380608/12495*:d^3Wm T: - 30888832/45815*d^5Wm
  (z-w)^-9  weight 21  -78567424/237405*:T T T Wm: - 107374592/870485*:dT dT Wm: - 34464768/20825*:dT dWm T: + 519793664/870485*:d^2T T Wm: - 742929408/4352425*:d^2T d^2Wm: + 308781824/687225*:d^3T dWm: - 1874944/137445*:d^4T Wm: - 998473216/4352425*:d^2Wm T T: + 221341952/69825*:d^3Wm dT: + 23129088/56525*:d^4Wm T: - 9058221952/13057275*d^6Wm
  (z-w)^-8  weight 22

  (z-w)^-2  weight 28  <88 terms>
  (z-w)^-1  weight 29  <117 terms>
W+(z) W+(w): 0 poles
W-(z) W-(w): 0 poles


## 8. Checks: no vacuum terms, nothing below weight $\Delta_0$, poles in the ideal of $W_{0,0}$

By [TW2013, Thm 5.4], every pole of every $W$–$W$ OPE lies in the ideal $X_{1,1}$ generated by $W_{0,0}$; in particular, it has no vacuum component and vanishes below weight $\Delta_0$. The cell below checks these two consequences directly. Setting `CHECK_IDEAL = True` also checks membership in the ideal of $\mathcal{M}_{p_+,p_-}$ generated by $W_{0,0}$, computed weight by weight by `M.ideal` (`06_subalgebras_ideals_quotients.ipynb`, section 3). This is expensive: it builds the weight spaces of the ideal up to weight $2\Delta_1-1$.

In [13]:
CHECK_IDEAL = False
ok = True
for poles in (ope_pm, ope_00, ope_0p, ope_0m, ope_pp, ope_mm):
    for n, x in enumerate(poles):
        w = 2 * Delta1 - n - 1
        ok = ok and x.coefficient(L._vacuum_key()) == 0
        ok = ok and (w >= Delta0 or x.is_zero())
print('no vacuum terms, no poles below weight Delta_0:', ok)
assert ok
if CHECK_IDEAL:
    X = M.ideal([W00_], names=['W00'], horizontal_grading='ambient')
    ok = all(x in X for poles in (ope_pm, ope_00, ope_0p, ope_0m, ope_pp, ope_mm)
             for x in poles)
    print('all W-W poles lie in the ideal generated by W00:', ok)
    assert ok

no vacuum terms, no poles below weight Delta_0: True


## 9. Normalisation

The $W$ fields are only defined up to nonzero constants, because of the normalisation of $P_\lambda$ in section 3. The pole of order $2\Delta_1 - \Delta_0$ of each $W$–$W$ OPE has weight $\Delta_0$ and is a multiple of $W_{0,0}$. The cell below computes these multiples: they are nonzero for $W^+W^-$ and $W^0W^0$, and zero for the other pairs, whose OPEs start at a higher weight. The two nonzero constants can be used to choose a normalisation of the $W$ fields.

In [14]:
def lowest_pole_constant(poles):
    n = 2 * Delta1 - 1 - Delta0
    if n >= len(poles) or poles[n].is_zero():
        return 0
    x = poles[n]
    k = next(iter(W00.support()))
    c = x.coefficient(k) / W00.coefficient(k)
    assert x == c * W00_
    return c

for name, poles in [('W+ W-', ope_pm), ('W0 W0', ope_00), ('W0 W+', ope_0p),
                    ('W0 W-', ope_0m), ('W+ W+', ope_pp), ('W- W-', ope_mm)]:
    print('%s: pole (z-w)^-%d = %s * W00' % (name, 2*Delta1 - Delta0, lowest_pole_constant(poles)))

W+ W-: pole (z-w)^-28 = 32768/2431 * W00
W0 W0: pole (z-w)^-28 = -131072/9163 * W00
W0 W+: pole (z-w)^-28 = 0 * W00
W0 W-: pole (z-w)^-28 = 0 * W00
W+ W+: pole (z-w)^-28 = 0 * W00
W- W-: pole (z-w)^-28 = 0 * W00


## 10. Limitations

* **Cost.** The strong generation check needs the spans of normally ordered monomials up to weight $2\Delta_1 - 1$, which grows quickly with $p_\pm$: it is $29$ for $(2,3)$, $53$ for $(2,5)$ and $69$ for $(3,4)$. In practice only the smallest pairs are within reach; the saved files make later runs cheap, but not the first one.
* **What is computed.** The notebook checks strong generation (one round of the closure), the weights, charges and primarity of the fields, and two consequences of [TW2013, Thm 5.4] (with `CHECK_IDEAL = True`, also membership in the ideal generated by $W_{0,0}$). That the subalgebra generated by $T, W^-, W^0, W^+$ is the intersection of the kernels of the screening operators is the theorem of [TW2013]; the kernels themselves are not computed.
* **Normalisation.** The $W$ fields are computed only up to nonzero constants (sections 3 and 9); no normalisation is chosen.
* **Modules.** The modules of $\mathcal M_{p_+,p_-}$ are not constructed here. The corresponding computations for $\mathcal W(2)$ are in `15_triplet_modules.ipynb`, whose limitations section explains what is missing for the modules of $\mathcal M_{2,3}$.
* **Saved files.** The saved state is a Sage pickle, tied to the parameters, the generators and the version of the library; a file that does not match is ignored and recomputed.

## References

- [FGST2006] B. L. Feigin, A. M. Gainutdinov, A. M. Semikhatov, I. Yu. Tipunin, *Logarithmic extensions of minimal models: characters and modular transformations*, Nuclear Phys. B 757 (2006), 303–343, [arXiv:hep-th/0606196](https://arxiv.org/abs/hep-th/0606196), [doi:10.1016/j.nuclphysb.2006.09.019](https://doi.org/10.1016/j.nuclphysb.2006.09.019). — the $(p_+,p_-)$ triplet algebras as kernels of two screening operators (introduction).
- [TW2013] A. Tsuchiya, S. Wood, *On the extended W-algebra of type $\mathfrak{sl}_2$ at positive rational level*, Int. Math. Res. Not. IMRN 2015 (2015), 5357–5435, [arXiv:1302.6435](https://arxiv.org/abs/1302.6435), [doi:10.1093/imrn/rnu090](https://doi.org/10.1093/imrn/rnu090). — the construction followed throughout: the dictionary, the screening operators, the $W$ fields, strong generation and the ideal generated by $W_{0,0}$ (sections 2–8).
- [Mac1995] I. G. Macdonald, *Symmetric functions and Hall polynomials*, 2nd ed., Oxford Math. Monogr., Oxford University Press (1995). — Jack symmetric functions (section 3).
- [GRW2009] M. R. Gaberdiel, I. Runkel, S. Wood, *Fusion rules and boundary conditions in the c = 0 triplet model*, J. Phys. A 42 (2009), 325403, [arXiv:0905.0916](https://arxiv.org/abs/0905.0916), [doi:10.1088/1751-8113/42/32/325403](https://doi.org/10.1088/1751-8113/42/32/325403). — the triplet algebra $\mathcal M_{2,3}$ at $c = 0$ (introduction).